# HDB resale prices × hawker centres: data preparation (revised)

Revised version of `Data Systems Task 2.ipynb`, aligned with the revised Task 1 design (`Task 1/revised/`).

**Inputs** (all under `data/raw/`):
- the resale file(s) listed in `RESALE_FILES`: here **Jan 2017 onwards**, which spans 2017-01 to 2026-09 and so contains a 3-year range for Q1
- `ListofGovernmentMarketsHawkerCentres.csv`
- `hawker_geo.csv`
- `hdb.csv` (HDB blocks with coordinates and postal codes)

The OneMap web service is used as a fallback.

**Outputs** (in `data/processed_steps/step_<STEP>/`): `town.csv`, `hdb_building.csv`, `hdb_flat.csv`, `resale_transaction.csv`, `hawker_centre.csv`, `proximity_level.csv`, `proximity.csv` and `data_quality_report.txt`.

No row is dropped because a value is missing. A missing value is either supplemented from another source or left NULL.

## Changes from the first version

| Change | Why |
|---|---|
| Paths point to `data/raw/...`; outputs go to `data/processed_steps/step_<STEP>/`; OneMap cache in `data/cache/` | Matches the repository layout; keeps the first version's outputs untouched |
| `RESALE_FILES` is a list; all listed files are combined before keys are assigned | Task 3 adds the other resale files one at a time; one run then gives consistent IDs |
| Missing `remaining_lease` column (files before 2015) gives NULL instead of stopping | Agreed design: store only what the source reports; Q2 derives remaining lease from `lease_commence_year` |
| `MULTI GENERATION` is mapped to `MULTI-GENERATION` | The 1990–1999 file uses a space; the schema allows only the hyphenated spelling |
| `straight_line_m` is no longer written to `proximity.csv` | It is derivable (`distance_m / 1.5`); storing both breaks 3NF |
| CSV headers are checked against the `\copy` column lists in `Task 1/revised/load_data.sql` | The check uses the real load script, not a copy of it |
| Validation also checks every Task 1 domain constraint and that each pair falls in exactly one level | Catches anything the database would reject before loading |

## Transformations

| Source | Transformation | Why |
|---|---|---|
| Resale | Trim whitespace; upper-case town, block, street, flat type, flat model | One spelling per value, so equal things group and join |
| Resale | `flat_type` `MULTI GENERATION` becomes `MULTI-GENERATION` | One spelling across files; matches the schema's CHECK |
| Resale | `month` "2017-01" becomes DATE 2017-01-01 | Matches the DATE column and sorts correctly |
| Resale | `storey_range` "10 TO 12" becomes `storey_lower` 10 and `storey_upper` 12 | Atomic numeric fields (1NF), range-queryable |
| Resale | `remaining_lease` becomes whole months ("61 years 04 months" gives 736; "70" gives 840); NULL if the file has no such column | One numeric unit; no invented values |
| Resale | `lease_commence_date` becomes integer `lease_commence_year` | Matches the schema |
| Resale | Split into town, hdb_building, hdb_flat, resale_transaction with surrogate keys | Removes repeated town, address and flat-profile data (BCNF) |
| Resale | Exact duplicate rows kept | No unit number separates them; they are separate sales |
| Join keys | Upper-case, keep only letters and digits | Punctuation and spacing differences cannot break a join; stored values keep their original text |
| Building coordinates | `hdb.csv` join on (block, street); OneMap for the rest; NULL if neither; source recorded | Offline and reproducible first, no invented values |
| Hawker files | Join on postal code (name-only links 17 of 107; postal code links 104; one manual match for Market Street) | Names differ between the files, postal codes do not |
| Government address | `S(731004)` becomes 731004; `S(141001/141002/141003)` becomes three codes | Several blocks can share one centre |
| Hawker centres in only one file | Kept; missing attributes are NULL | The NULL policy |
| Hawker coordinates for unmatched centres | `hdb.csv` by postal code, then OneMap, then `hawker_supplement.csv`, then NULL | Supplement before leaving NULL |
| Stall counts | `total_stalls` checked against cooked + produce | Consistency check on the source |
| Proximity | Haversine straight-line distance, then estimated walking distance = 1.5 × straight-line; pairs under 2,400 m walking stored; three levels (0–0.8, 0.8–1.6, 1.6–2.4 km) in a lookup table | Real paths are longer than straight lines; levels use the walking estimate |

In [1]:
%pip install -q pandas numpy requests

Note: you may need to restart the kernel to use updated packages.


## Initial setup
`ROOT` is found by walking up from the notebook's folder to the first folder that contains `data/raw`, so the notebook runs from any working directory inside the repository.

In [1]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# No row is dropped because a value is missing. A missing value is
# either supplemented from another source or left NULL (an empty field in the CSV).

# repository root: the first folder, going upwards, that contains data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
RAW = ROOT / "data" / "raw"
# Task 3 database-size step: step k combines the first k files of ALL_RESALE_FILES.
# Set it here, or from outside with the environment variable RESALE_STEP.
STEP = int(os.environ.get("RESALE_STEP", 1))

# Resale files in the order Task 3 adds them, one per step.
RESALE_DIR = RAW / "flat_resale_prices"
ALL_RESALE_FILES = [
    RESALE_DIR / "Resale flat prices based on registration date from Jan-2017 onwards.csv",                  # step 1
    RESALE_DIR / "Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",           # step 2
    RESALE_DIR / "Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",           # step 3
    RESALE_DIR / "Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",                         # step 4
    RESALE_DIR / "Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv",                             # step 5
]
if not 1 <= STEP <= len(ALL_RESALE_FILES):
    raise ValueError(f"STEP must be 1-{len(ALL_RESALE_FILES)}, got {STEP}")
RESALE_FILES = ALL_RESALE_FILES[:STEP]

# one output folder per step: data/processed_steps/step_1 ... step_5
OUT = ROOT / "data" / "processed_steps" / f"step_{STEP}"
OUT.mkdir(parents=True, exist_ok=True)
GOV_FILE = RAW / "hawker_centres" / "ListofGovernmentMarketsHawkerCentres.csv"
GEO_FILE = RAW / "hawker_geo" / "hawker_geo.csv"
HDB_FILE = RAW / "hdb_geo" / "hdb.csv"                       # supplementary: HDB blocks with coordinates + postal codes
CACHE_FILE = ROOT / "data" / "cache" / "onemap_search_cache.json"   # OneMap answers already fetched
SUPPLEMENT_FILE = RAW / "hawker_supplement.csv"              # optional hand-typed values; fills blanks only
LOAD_SCRIPT = ROOT / "Task 1" / "revised" / "load_data.sql"  # COPY column lists the CSVs must match

DELAY = 0.3     # seconds between OneMap calls
# os.environ["ONEMAP_TOKEN"] = "paste-your-token-here"   # only if OneMap asks for authentication

# Proximity. Levels apply to the ESTIMATED WALKING distance = WALK_RATIO x straight-line distance.
WALK_RATIO = 1.5        # real roads and paths are longer than straight lines
LEVELS = [(1, "Walking distance", 0, 800),       # 0 - 0.8 km
          (2, "Nearby", 800, 1600),              # 0.8 - 1.6 km
          (3, "Accessible", 1600, 2400)]         # 1.6 - 2.4 km
MAX_DIST_M = LEVELS[-1][3]   # pairs whose estimated walking distance is at or beyond this are not stored (2400)
CROSSCHECK_TOL_M = 100  # hdb.csv vs OneMap coordinates further apart than this are flagged
LAT_RANGE, LON_RANGE = (1.15, 1.48), (103.6, 104.1)   # sanity box around Singapore (same as the schema)
# government-list name -> coordinates-file name, for centres the postal-code join cannot link
MANUAL_MATCH = {"Market Street Food Centre": "Market Street Hawker Centre"}
# spellings that differ between resale files -> the spelling the schema allows
FLAT_TYPE_FIX = {"MULTI GENERATION": "MULTI-GENERATION"}
FLAT_TYPES = {"1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"}

ONEMAP_SEARCH = "https://www.onemap.gov.sg/api/common/elastic/search"


def die(msg):
    raise RuntimeError(msg)


print("step:", STEP, "| resale files:", len(RESALE_FILES))
print("repository root:", ROOT)
print("output folder:  ", OUT)
for f in (*RESALE_FILES, GOV_FILE, GEO_FILE, HDB_FILE, LOAD_SCRIPT):
    print("found  " if f.exists() else "MISSING", f.relative_to(ROOT))
print(f"proximity cutoff: {MAX_DIST_M} m estimated walking = about {MAX_DIST_M / WALK_RATIO:.0f} m straight-line")

repository root: /Users/kuberbellad04/Documents/MSDS/Sem 1/Data Systems/hexa_nexus
output folder:   /Users/kuberbellad04/Documents/MSDS/Sem 1/Data Systems/hexa_nexus/data/processed_revised
found   data/raw/flat_resale_prices/Resale flat prices based on registration date from Jan-2017 onwards.csv
found   data/raw/hawker_centres/ListofGovernmentMarketsHawkerCentres.csv
found   data/raw/hawker_geo/hawker_geo.csv
found   data/raw/hdb_geo/hdb.csv
found   Task 1/revised/load_data.sql
proximity cutoff: 2400 m estimated walking = about 1600 m straight-line


## Helper functions
- **OneMap helpers:** `onemap_search` and `cached_search`. Every response is cached in `data/cache/`, so a query is never asked twice.
- **Street matching:** `norm_street` expands abbreviations ("AVE" to "AVENUE").
- **`join_key`:** upper-case, letters and digits only.
- **`haversine_m`:** straight-line (great-circle) distance.
- **`read_hdb_file`:** reads `hdb.csv` with postal codes as strings, so leading zeroes are kept (postal codes have 6 digits).

In [2]:
def load_cache():
    # answers already fetched from OneMap, so a query is never asked twice
    return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}


def save_cache(cache):
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    tmp = CACHE_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(cache))
    tmp.replace(CACHE_FILE)


def onemap_search(query, session, retries=4):
    """Ask OneMap's search endpoint for one free-text query; returns its list of matches."""
    headers = {}
    token = os.environ.get("ONEMAP_TOKEN")
    if token:
        headers["Authorization"] = token
    params = {"searchVal": query, "returnGeom": "Y", "getAddrDetails": "Y", "pageNum": 1}
    for attempt in range(retries):
        try:
            r = session.get(ONEMAP_SEARCH, params=params, headers=headers, timeout=20)
            if r.status_code == 200:
                return r.json().get("results", [])
            if r.status_code in (401, 403):
                die(f"OneMap returned HTTP {r.status_code}. Create a free OneMap account and "
                    "set ONEMAP_TOKEN in the setup cell, then rerun (progress so far is cached).")
        except requests.RequestException:
            pass                      # network hiccup: fall through and retry
        time.sleep(2 ** attempt)      # wait 1, 2, 4, 8 seconds between retries
    raise RuntimeError(f"OneMap search kept failing for {query!r}")


def cached_search(query, cache, session, delay, retry_empty=False):
    """OneMap lookup with a cache. retry_empty=True asks again if the cached answer was empty."""
    if query not in cache or (retry_empty and not cache[query]):
        res = onemap_search(query, session)
        cache[query] = [
            {k: x.get(k) for k in ("BLK_NO", "ROAD_NAME", "POSTAL", "ADDRESS", "LATITUDE", "LONGITUDE")}
            for x in res[:5]
        ]
        time.sleep(delay)             # be polite to the server
    return cache[query]


ABBR = {
    "AVE": "AVENUE", "ST": "STREET", "RD": "ROAD", "DR": "DRIVE", "CRES": "CRESCENT", "BT": "BUKIT",
    "JLN": "JALAN", "LOR": "LORONG", "CTRL": "CENTRAL", "NTH": "NORTH", "STH": "SOUTH", "CL": "CLOSE",
    "PL": "PLACE", "TER": "TERRACE", "GDNS": "GARDENS", "HTS": "HEIGHTS", "PK": "PARK", "UPP": "UPPER",
    "TG": "TANJONG", "KG": "KAMPONG", "C'WEALTH": "COMMONWEALTH", "MKT": "MARKET", "CTR": "CENTRE",
    "LK": "LINK", "WK": "WALK", "BLVD": "BOULEVARD",
}


def norm_street(s):
    # expand abbreviations so "ANG MO KIO AVE 4" equals OneMap's "ANG MO KIO AVENUE 4"
    s = re.sub(r"\bST\.? GEORGE", "SAINT GEORGE", str(s).upper())
    s = re.sub(r"[.,]", " ", s)
    return " ".join(ABBR.get(t, t) for t in s.split())


def join_key(value):
    """Matching key only: upper-case, letters and digits only. '4A, Woodlands Ctr Rd' -> '4AWOODLANDSCTRRD'."""
    return re.sub(r"[^A-Z0-9]", "", str(value).upper()) if pd.notna(value) else ""


def building_query(block, street):
    return f"{block} {street}"


def postal_codes(text):
    """All 6-digit postal codes in a text: 'S(141001/141002)' -> ['141001', '141002']."""
    return re.findall(r"\b\d{6}\b", str(text))


def pick_result(block, street, results):
    """Choose the OneMap hit for a building. exact = block and street both match."""
    cands = [r for r in results if str(r.get("BLK_NO") or "").upper() == str(block).upper()]
    ns = norm_street(street)
    exact = [r for r in cands if norm_street(r.get("ROAD_NAME") or "") == ns]
    if exact:
        return exact[0], "exact"
    if cands:
        return cands[0], "block_only"
    return None, "failed"


def in_singapore(lat, lon):
    return LAT_RANGE[0] <= lat <= LAT_RANGE[1] and LON_RANGE[0] <= lon <= LON_RANGE[1]


def haversine_m(lat1, lon1, lat2, lon2):
    """Straight-line (great-circle) distance in metres. Inputs in degrees; numpy broadcasting applies."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlmb = p2 - p1, np.radians(lon2) - np.radians(lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * 6371000.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def read_hdb_file():
    """hdb.csv -> block_key, street_key, postal, latitude, longitude. Returns None if the file is absent."""
    if not HDB_FILE.exists():
        return None
    h = pd.read_csv(HDB_FILE, dtype={"blk_no": str, "street": str, "postal": str})
    h["block_key"] = h["blk_no"].map(join_key)
    h["street_key"] = h["street"].map(join_key)
    h["postal"] = h["postal"].str.strip().str.zfill(6)     # keep leading zeros: 080001
    h["latitude"] = pd.to_numeric(h["lat"], errors="coerce")
    h["longitude"] = pd.to_numeric(h["lng"], errors="coerce")
    usable = h["latitude"].between(*LAT_RANGE) & h["longitude"].between(*LON_RANGE)
    return h.loc[usable, ["block_key", "street_key", "postal", "latitude", "longitude"]]

## Step 1: profile the raw files
Before changing anything, we look at each input (size, columns, NULLs and a few rows) to see what the data looks like and what problems to expect. For example, `hawker_geo.csv` has 17 blank addresses, and files before 2015 have no `remaining_lease` column.

In [3]:
def profile(path, n=3):
    df = pd.read_csv(path, low_memory=False)
    print("=" * 70)
    print(f"{path.name}: {len(df):,} rows x {df.shape[1]} columns")
    print("columns:", list(df.columns))
    nulls = df.isna().sum()
    print("columns containing NULLs:", nulls[nulls > 0].to_dict() or "none")
    display(df.head(n))


for f in (*RESALE_FILES, GOV_FILE, GEO_FILE, HDB_FILE):
    if f.exists():
        profile(f)

Resale flat prices based on registration date from Jan-2017 onwards.csv: 241,483 rows x 11 columns
columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']
columns containing NULLs: none


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0


ListofGovernmentMarketsHawkerCentres.csv: 107 rows x 7 columns
columns: ['name_of_centre', 'location_of_centre', 'type_of_centre', 'owner', 'no_of_stalls', 'no_of_cooked_food_stalls', 'no_of_mkt_produce_stalls']
columns containing NULLs: none


,name_of_centre,location_of_centre,type_of_centre,owner,no_of_stalls,no_of_cooked_food_stalls,no_of_mkt_produce_stalls
0,Adam Road Food Centre,"2, Adam Road, S(289876)",HC,Government,32,32,0
1,Amoy Street Food Centre,"National Development Building, Annex B, Telok ...",HC,Government,135,134,1
2,Bedok Food Centre,"1, Bedok Road, S(469572)",HC,Government,32,32,0


hawker_geo.csv: 129 rows x 4 columns
columns: ['longitude', 'latitude', 'NAME', 'ADDRESS_MYENV']
columns containing NULLs: {'ADDRESS_MYENV': 17}


,longitude,latitude,NAME,ADDRESS_MYENV
0,103.800367,1.306900,Commonwealth Crescent Market,"31, Commonwealth Crescent, Singapore 149644"
1,103.832182,1.284786,Tiong Bahru Market,"30, Seng Poh Road, Singapore 168898"
2,103.863878,1.303142,Golden Mile Food Centre,"505, Beach Road, Singapore 199583"


hdb.csv: 12,442 rows x 37 columns
columns: ['Unnamed: 0', 'blk_no', 'street', 'max_floor_lvl', 'year_completed', 'residential', 'commercial', 'market_hawker', 'miscellaneous', 'multistorey_carpark', 'precinct_pavilion', 'bldg_contract_town', 'total_dwelling_units', '1room_sold', '2room_sold', '3room_sold', '4room_sold', '5room_sold', 'exec_sold', 'multigen_sold', 'studio_apartment_sold', '1room_rental', '2room_rental', '3room_rental', 'other_room_rental', 'lat', 'lng', 'building', 'addr', 'postal', 'SUBZONE_NO', 'SUBZONE_N', 'SUBZONE_C', 'PLN_AREA_N', 'PLN_AREA_C', 'REGION_N', 'REGION_C']
columns containing NULLs: {'SUBZONE_NO': 1, 'SUBZONE_N': 1, 'SUBZONE_C': 1, 'PLN_AREA_N': 1, 'PLN_AREA_C': 1, 'REGION_N': 1, 'REGION_C': 1}


,Unnamed: 0,blk_no,street,max_floor_lvl,year_completed,residential,commercial,market_hawker,miscellaneous,multistorey_carpark,...,building,addr,postal,SUBZONE_NO,SUBZONE_N,SUBZONE_C,PLN_AREA_N,PLN_AREA_C,REGION_N,REGION_C
0,0,1,BEACH RD,16,1970,Y,Y,N,N,N,...,RAFFLES HOTEL,1 BEACH ROAD RAFFLES HOTEL SINGAPORE 189673,189673,2.0,CITY HALL,DTSZ02,DOWNTOWN CORE,DT,CENTRAL REGION,CR
1,1,1,BEDOK STH AVE 1,14,1975,Y,N,N,Y,N,...,NIL,1 BEDOK SOUTH AVENUE 1 SINGAPORE 460001,460001,6.0,BEDOK SOUTH,BDSZ06,BEDOK,BD,EAST REGION,ER
2,2,1,CANTONMENT RD,2,2010,N,Y,N,N,N,...,PINNACLE @ DUXTON,1 CANTONMENT ROAD PINNACLE @ DUXTON SINGAPORE ...,080001,3.0,CHINATOWN,OTSZ03,OUTRAM,OT,CENTRAL REGION,CR


## Step 2: resale file(s) to the town, building, flat and transaction tables
All files in `RESALE_FILES` are read and combined first, so the surrogate keys are assigned once over the whole set.

Cleaning:
- Trim whitespace in every value; upper-case town, block, street, flat type and flat model.
- Map `MULTI GENERATION` to `MULTI-GENERATION`.
- Convert `month` to a real date.
- Split `storey_range` into a lower and an upper floor.
- Convert `remaining_lease` to months. A file without that column (before 2015) gives NULL.

Then the data is split into four tables with surrogate keys:
- **town:** one row per town.
- **hdb_building:** one row per (town, block, street).
- **hdb_flat:** one row per distinct flat profile within a building.
- **resale_transaction:** one row per sale.

Checks: required values must parse, flat types must be valid, merges are validated, and no (block, street) address may appear in two towns.

In [4]:
def parse_remaining_lease(v):
    """'61 years 04 months' -> 736, '61 years' -> 732, plain '70' (whole years) -> 840."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().lower()
    y = re.search(r"(\d+)\s*year", s)
    m = re.search(r"(\d+)\s*month", s)
    if y or m:
        return (int(y.group(1)) if y else 0) * 12 + (int(m.group(1)) if m else 0)
    if re.fullmatch(r"\d+", s):
        return int(s) * 12
    return np.nan


def read_resale(path):
    """One resale file -> DataFrame of strings. A missing remaining_lease column becomes NULL."""
    path = Path(path)
    if not path.exists():
        die(f"resale file not found: {path}")
    df = pd.read_csv(path, dtype=str)
    for c in df.columns:
        df[c] = df[c].str.strip()
    need = ["month", "town", "flat_type", "block", "street_name", "storey_range", "floor_area_sqm",
            "flat_model", "lease_commence_date", "resale_price"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        die(f"{path.name} lacks columns {miss}")
    if "remaining_lease" not in df.columns:
        df["remaining_lease"] = np.nan        # not reported before 2015: stays NULL
        note = "no remaining_lease column -> NULL"
    else:
        note = "remaining_lease present"
    print(f"read {len(df):,} rows from {path.name} ({df['month'].min()} to {df['month'].max()}; {note})")
    return df[need + ["remaining_lease"]]


def stage_resale(paths):
    df = pd.concat([read_resale(p) for p in paths], ignore_index=True)
    print(f"combined: {len(df):,} rows from {len(paths)} file(s)")
    print(f"exact duplicate rows (all kept: no unit number separates them): {df.duplicated().sum():,}")

    for c in ["town", "block", "street_name", "flat_type", "flat_model"]:
        df[c] = df[c].str.upper()
    fixed = df["flat_type"].isin(FLAT_TYPE_FIX.keys())
    df["flat_type"] = df["flat_type"].replace(FLAT_TYPE_FIX)
    print(f"flat_type spellings normalised ({FLAT_TYPE_FIX}): {int(fixed.sum())} rows")
    bad_type = ~df["flat_type"].isin(FLAT_TYPES)
    if bad_type.any():
        die(f"unknown flat_type values: {sorted(df.loc[bad_type, 'flat_type'].unique())}")

    multi_town = (df.groupby(["block", "street_name"])["town"].nunique() > 1).sum()
    print(f"(block, street) addresses found in more than one town: {int(multi_town)}")
    if multi_town:
        die("an address maps to two towns; the building -> town dependency would break")

    df["transaction_month"] = pd.to_datetime(df["month"] + "-01", format="%Y-%m-%d", errors="coerce")
    st = df["storey_range"].str.extract(r"(\d+)\s*TO\s*(\d+)", flags=re.IGNORECASE)
    df["storey_lower"] = pd.to_numeric(st[0], errors="coerce")
    df["storey_upper"] = pd.to_numeric(st[1], errors="coerce")
    df["floor_area_sqm"] = pd.to_numeric(df["floor_area_sqm"], errors="coerce")
    df["lease_commence_year"] = pd.to_numeric(df["lease_commence_date"], errors="coerce")
    df["resale_price"] = pd.to_numeric(df["resale_price"], errors="coerce")
    df["reported_remaining_lease_months"] = pd.to_numeric(
        df["remaining_lease"].map(parse_remaining_lease), errors="coerce")

    required = ["transaction_month", "town", "block", "street_name", "flat_type", "flat_model",
                "floor_area_sqm", "storey_lower", "storey_upper", "lease_commence_year", "resale_price"]
    bad = df[required].isna().any(axis=1)
    if bad.any():
        print(df.loc[bad].head(10).to_string())
        die(f"{int(bad.sum())} rows have an unparseable required value (sample above).")
    # NULL is expected only where the source file has no remaining_lease column
    unparsed = df["remaining_lease"].notna() & df["reported_remaining_lease_months"].isna()
    if unparsed.any():
        die(f"{int(unparsed.sum())} remaining_lease values could not be parsed, e.g. "
            f"{df.loc[unparsed, 'remaining_lease'].unique()[:5].tolist()}")
    print(f"remaining_lease NULL (not reported by the source): {int(df['reported_remaining_lease_months'].isna().sum()):,}")
    for c in ["storey_lower", "storey_upper", "lease_commence_year"]:
        df[c] = df[c].astype(int)

    df["transaction_id"] = np.arange(1, len(df) + 1)

    towns = sorted(df["town"].unique())
    town_df = pd.DataFrame({"town_id": range(1, len(towns) + 1), "town_name": towns})
    town_id = dict(zip(town_df["town_name"], town_df["town_id"]))

    bkeys = ["town", "block", "street_name"]
    bld = df[bkeys].drop_duplicates().sort_values(bkeys).reset_index(drop=True)
    bld["building_id"] = bld.index + 1
    bld["town_id"] = bld["town"].map(town_id)
    df = df.merge(bld[bkeys + ["building_id"]], on=bkeys, how="left", validate="many_to_one")

    fkeys = ["building_id", "flat_type", "flat_model", "floor_area_sqm", "storey_lower",
             "storey_upper", "lease_commence_year"]
    flats = df[fkeys].drop_duplicates().sort_values(fkeys).reset_index(drop=True)
    flats["flat_id"] = flats.index + 1
    df = df.merge(flats, on=fkeys, how="left", validate="many_to_one").sort_values("transaction_id")
    assert df["building_id"].notna().all() and df["flat_id"].notna().all()

    town_df.to_csv(OUT / "town.csv", index=False)
    bld[["building_id", "town_id", "block", "street_name"]].to_csv(OUT / "building_base.csv", index=False)
    flats[["flat_id", "building_id", "flat_type", "flat_model", "floor_area_sqm", "storey_lower",
           "storey_upper", "lease_commence_year"]].to_csv(OUT / "hdb_flat.csv", index=False)
    tx = df[["transaction_id", "flat_id", "transaction_month", "resale_price",
             "reported_remaining_lease_months"]].copy()
    tx["transaction_month"] = tx["transaction_month"].dt.strftime("%Y-%m-%d")
    tx["reported_remaining_lease_months"] = tx["reported_remaining_lease_months"].astype("Int64")
    tx.to_csv(OUT / "resale_transaction.csv", index=False)
    print(f"towns: {len(town_df)}  buildings: {len(bld):,}  flat profiles: {len(flats):,}  "
          f"transactions: {len(tx):,}")


stage_resale(RESALE_FILES)

read 241,483 rows from Resale flat prices based on registration date from Jan-2017 onwards.csv (2017-01 to 2026-09; remaining_lease present)
combined: 241,483 rows from 1 file(s)
exact duplicate rows (all kept: no unit number separates them): 318
flat_type spellings normalised ({'MULTI GENERATION': 'MULTI-GENERATION'}): 0 rows
(block, street) addresses found in more than one town: 0
remaining_lease NULL (not reported by the source): 0
towns: 26  buildings: 9,753  flat profiles: 98,205  transactions: 241,483


## Step 3: building coordinates
The resale files have addresses, not coordinates, so two sources are used:
1. **`hdb.csv`:** a local join on (block, street).
2. **OneMap:** only for buildings not found in `hdb.csv`.

If neither source has a building, its coordinates stay NULL and `coordinate_source` is `none`.

For buildings located by both sources (using cached OneMap answers only), we measure how far apart the two place them. A small median gap supports the data; a large one is a red flag.

In [5]:
def stage_coordinates(delay):
    base_path = OUT / "building_base.csv"
    if not base_path.exists():
        die("run the resale stage (Step 2) first")
    base = pd.read_csv(base_path, dtype={"block": str, "street_name": str})
    base["block_key"] = base["block"].map(join_key)
    base["street_key"] = base["street_name"].map(join_key)
    base["latitude"], base["longitude"], base["coordinate_source"] = np.nan, np.nan, "none"

    # Source 1: hdb.csv (a local join: fast and identical on every run)
    hdb = read_hdb_file()
    if hdb is None:
        print("hdb.csv not found: every building goes to OneMap (about an hour the first time)")
    else:
        keys = ["block_key", "street_key"]
        spread = hdb.groupby(keys)[["latitude", "longitude"]].nunique().gt(1).any(axis=1).sum()
        print(f"addresses with more than one coordinate in hdb.csv (first one used): {int(spread)}")
        lookup = hdb.drop_duplicates(keys)[keys + ["latitude", "longitude"]]
        base = (base.drop(columns=["latitude", "longitude"])
                    .merge(lookup, on=keys, how="left", validate="many_to_one"))
        found = base["latitude"].notna()
        base.loc[found, "coordinate_source"] = "hdb_dataset"
        print(f"found in hdb.csv: {int(found.sum()):,} of {len(base):,} buildings ({found.mean():.2%})")

    # Source 2: OneMap, only for buildings that still have no coordinates (answers are cached)
    cache, session = load_cache(), requests.Session()
    queries = {i: building_query(r["block"], r["street_name"])
               for i, r in base[base["latitude"].isna()].iterrows()}
    todo = [q for q in dict.fromkeys(queries.values()) if q not in cache]
    if todo:
        print(f"{len(todo):,} buildings to look up on OneMap (about {len(todo) * (delay + 0.3) / 60:.0f} min). "
              "Interrupting is safe; rerun to resume.")
    try:
        for n, q in enumerate(todo, 1):
            cached_search(q, cache, session, delay)
            if n % 50 == 0:
                save_cache(cache)
                print(f"  {n:,}/{len(todo):,}")
    finally:
        save_cache(cache)
    for i, q in queries.items():
        hit, status = pick_result(base.at[i, "block"], base.at[i, "street_name"], cache.get(q, []))
        if not hit:
            continue
        try:
            lat, lon = float(hit["LATITUDE"]), float(hit["LONGITUDE"])
        except (TypeError, ValueError):
            continue
        if in_singapore(lat, lon):
            base.at[i, "latitude"], base.at[i, "longitude"] = round(lat, 6), round(lon, 6)
            base.at[i, "coordinate_source"] = f"onemap_{status}"     # onemap_exact / onemap_block_only

    # Cross-check: buildings located by hdb.csv AND answered by OneMap (cached answers only, no network)
    gaps = []
    for i in base.index[base["coordinate_source"] == "hdb_dataset"]:
        q = building_query(base.at[i, "block"], base.at[i, "street_name"])
        hit, status = pick_result(base.at[i, "block"], base.at[i, "street_name"], cache.get(q, []))
        if status != "exact":
            continue
        try:
            gaps.append(float(haversine_m(base.at[i, "latitude"], base.at[i, "longitude"],
                                          float(hit["LATITUDE"]), float(hit["LONGITUDE"]))))
        except (TypeError, ValueError):
            continue
    if gaps:
        g = np.array(gaps)
        print(f"cross-check hdb.csv vs OneMap on {len(g):,} buildings: median gap {np.median(g):.0f} m, "
              f"95th percentile {np.percentile(g, 95):.0f} m, "
              f"{int((g > CROSSCHECK_TOL_M).sum())} differ by more than {CROSSCHECK_TOL_M} m")

    out = base[["building_id", "town_id", "block", "street_name", "latitude", "longitude", "coordinate_source"]].copy()
    out[["latitude", "longitude"]] = out[["latitude", "longitude"]].round(6)
    out.to_csv(OUT / "hdb_building.csv", index=False)
    print(out["coordinate_source"].value_counts().to_string())
    none = out[out["coordinate_source"] == "none"]
    if len(none):
        print("buildings no source could locate (coordinates stay NULL), first 20:")
        print(none[["building_id", "block", "street_name"]].head(20).to_string(index=False))


stage_coordinates(DELAY)

addresses with more than one coordinate in hdb.csv (first one used): 0
found in hdb.csv: 9,752 of 9,753 buildings (99.99%)
cross-check hdb.csv vs OneMap on 7,900 buildings: median gap 0 m, 95th percentile 6 m, 17 differ by more than 100 m
coordinate_source
hdb_dataset     9752
onemap_exact       1


## Step 4: merge the two hawker files
`hawker_geo.csv` has coordinates but lacks other details. The government list has stall details but no coordinates.

**Join key: postal code, not name.** The code prints how many centres a name-only join would link.

For the centres `hawker_geo.csv` lacks, coordinates come from, in order:
1. `hdb.csv`, looked up by postal code.
2. OneMap, only if the postal code, or both road and block, match.
3. `hawker_supplement.csv`.
4. NULL.

Centres that appear in only one file are all retained. Unknown attributes are NULL. `data_source` records where each row came from.

In [6]:
def clean_gov_address(s):
    return re.sub(r"S\((\d{6}(?:/\d{6})*)\)", r"Singapore \1", re.sub(r"\s+", " ", str(s))).strip()


def street_query(location):
    """'Blk 4A, Woodlands Centre Road, S(731004)' -> '4A Woodlands Centre Road' (a plain address query)."""
    s = re.sub(r"S\([\d/]+\)", "", str(location))
    s = re.sub(r"\bBlks?\b\.?", "", s, flags=re.IGNORECASE)
    s = s.replace(",", " ")
    return re.sub(r"\s+", " ", s).strip()


def coordinates_from_hdb(location, hdb_by_postal):
    """Government-only centres sit in HDB blocks: look their postal codes up in hdb.csv (mean if several)."""
    pts = [hdb_by_postal[p] for p in postal_codes(location) if p in hdb_by_postal]
    if not pts:
        return np.nan, np.nan
    return float(np.mean([p[0] for p in pts])), float(np.mean([p[1] for p in pts]))


def coordinates_from_onemap(row, cache, session, delay):
    """Try each postal code, then the street address. Accept only a hit that plausibly is this place."""
    location = str(row["location_of_centre"])
    for q in postal_codes(location) + [street_query(location)]:
        if not q:
            continue
        for hit in cached_search(q, cache, session, delay, retry_empty=True):
            if q.isdigit():                        # postal-code query: the hit must carry that postal code
                ok = str(hit.get("POSTAL")) == q
            else:                                  # address query: road name and block number must both appear
                road = norm_street(hit.get("ROAD_NAME") or "")
                blk = str(hit.get("BLK_NO") or "").upper()
                ok = bool(road) and road in norm_street(location) and \
                     bool(blk) and re.search(rf"\b{re.escape(blk)}\b", location.upper()) is not None
            if not ok:
                continue
            try:
                lat, lon = float(hit["LATITUDE"]), float(hit["LONGITUDE"])
            except (TypeError, ValueError):
                continue
            if in_singapore(lat, lon):
                return lat, lon
    return np.nan, np.nan


def stage_hawkers(delay):
    gov = pd.read_csv(GOV_FILE)
    geo = pd.read_csv(GEO_FILE)
    gov["name"] = gov["name_of_centre"].str.replace(r"\s+", " ", regex=True).str.strip()
    geo["name"] = geo["NAME"].str.replace(r"\s+", " ", regex=True).str.strip()

    by_name = gov["name"].map(join_key).isin(set(geo["name"].map(join_key))).sum()
    print(f"joining by name alone would link {int(by_name)} of {len(gov)} centres, so we join by postal code")

    pc_to_geo, shared = {}, 0
    for i, addr in geo["ADDRESS_MYENV"].items():
        for p in postal_codes(addr):
            shared += p in pc_to_geo
            pc_to_geo.setdefault(p, i)
    print(f"postal codes used by more than one row of hawker_geo.csv: {shared}")

    match = {}
    for gi, loc in gov["location_of_centre"].items():
        hits = {pc_to_geo[p] for p in postal_codes(loc) if p in pc_to_geo}
        if hits:
            match[gi] = min(hits)
    print(f"joined by postal code: {len(match)} of {len(gov)}")
    for gname, geo_name in MANUAL_MATCH.items():
        gi = gov.index[gov["name"] == gname]
        gj = geo.index[geo["name"] == geo_name]
        if len(gi) == 1 and len(gj) == 1 and gi[0] not in match:
            match[gi[0]] = gj[0]
    dup = pd.Series(list(match.values())).duplicated().sum()
    if dup:
        print(f"WARNING: {dup} coordinates rows are matched by more than one government row")
    print(f"after MANUAL_MATCH: {len(match)} of {len(gov)}")

    cache, session = load_cache(), requests.Session()
    hdb = read_hdb_file()
    hdb_by_postal = {} if hdb is None else {
        p: (a, b) for p, a, b in zip(hdb["postal"], hdb["latitude"], hdb["longitude"]) if pd.notna(p)}
    rows, used_geo = [], set()
    try:
        for gi, g in gov.iterrows():
            addr_gov = clean_gov_address(g["location_of_centre"])
            if gi in match:
                gr = geo.loc[match[gi]]
                used_geo.add(match[gi])
                lat, lon = float(gr["latitude"]), float(gr["longitude"])
                addr = gr["ADDRESS_MYENV"] if pd.notna(gr["ADDRESS_MYENV"]) else addr_gov
                src = "gov+geo"
            else:
                lat, lon = coordinates_from_hdb(g["location_of_centre"], hdb_by_postal)
                if pd.notna(lat):
                    src = "gov+hdb"
                else:
                    lat, lon = coordinates_from_onemap(g, cache, session, delay)
                    src = "gov+onemap" if pd.notna(lat) else "gov_only"
                addr = addr_gov
                print(f"  not in hawker_geo.csv: '{g['name']}' -> {src}")
            rows.append({"centre_name": g["name"], "address": addr, "centre_type": g["type_of_centre"],
                         "latitude": lat, "longitude": lon, "total_stalls": g["no_of_stalls"],
                         "cooked_food_stalls": g["no_of_cooked_food_stalls"],
                         "market_produce_stalls": g["no_of_mkt_produce_stalls"], "data_source": src})
    finally:
        save_cache(cache)

    # centres that appear only in hawker_geo.csv: kept, with unknown attributes left NULL
    for gj, gr in geo.iterrows():
        if gj in used_geo:
            continue
        rows.append({"centre_name": gr["name"],
                     "address": gr["ADDRESS_MYENV"] if pd.notna(gr["ADDRESS_MYENV"]) else None,
                     "centre_type": None, "latitude": float(gr["latitude"]), "longitude": float(gr["longitude"]),
                     "total_stalls": None, "cooked_food_stalls": None, "market_produce_stalls": None,
                     "data_source": "geo_only"})
    df = pd.DataFrame(rows)
    for c in ["latitude", "longitude", "total_stalls", "cooked_food_stalls", "market_produce_stalls"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # optional hand-typed values (fill blanks only; never overwrite)
    if SUPPLEMENT_FILE.exists():
        sup = pd.read_csv(SUPPLEMENT_FILE).drop_duplicates("centre_name").set_index("centre_name")
        for c in ["address", "centre_type", "latitude", "longitude",
                  "cooked_food_stalls", "market_produce_stalls", "total_stalls"]:
            if c in sup.columns:
                fill = df["centre_name"].map(sup[c])
                if c not in ("address", "centre_type"):
                    fill = pd.to_numeric(fill, errors="coerce")
                df[c] = df[c].where(df[c].notna(), fill)
        manual = (df["data_source"] == "gov_only") & df["latitude"].notna() & df["longitude"].notna()
        df.loc[manual, "data_source"] = "gov+manual"
        print(f"applied supplement file ({len(sup)} rows); only blanks were filled")

    # total_stalls is derivable: fill it when the other two are known
    m = df["total_stalls"].isna() & df["cooked_food_stalls"].notna() & df["market_produce_stalls"].notna()
    df.loc[m, "total_stalls"] = df.loc[m, "cooked_food_stalls"] + df.loc[m, "market_produce_stalls"]

    if df["centre_name"].duplicated().any():
        die(f"duplicate centre names: {df.loc[df['centre_name'].duplicated(), 'centre_name'].tolist()}")
    chk = df.dropna(subset=["total_stalls", "cooked_food_stalls", "market_produce_stalls"])
    bad = chk[chk["total_stalls"] != chk["cooked_food_stalls"] + chk["market_produce_stalls"]]
    print(f"rows where total != cooked + produce: {len(bad)}")

    df = df.sort_values("centre_name").reset_index(drop=True)
    df["hawker_id"] = df.index + 1
    df[["latitude", "longitude"]] = df[["latitude", "longitude"]].round(6)
    for c in ["total_stalls", "cooked_food_stalls", "market_produce_stalls"]:
        df[c] = df[c].astype("Int64")
    cols = ["hawker_id", "centre_name", "address", "centre_type", "latitude", "longitude",
            "total_stalls", "cooked_food_stalls", "market_produce_stalls", "data_source"]
    df[cols].to_csv(OUT / "hawker_centre.csv", index=False)
    print(df["data_source"].value_counts().to_string())
    print(f"hawker centres: {len(df)}")
    no_coords = df[df["latitude"].isna() | df["longitude"].isna()]
    if len(no_coords):
        print("\nCentres with NULL coordinates (left out of proximity until coordinates are supplied):")
        print(no_coords[["hawker_id", "centre_name"]].to_string(index=False))


stage_hawkers(DELAY)

joining by name alone would link 17 of 107 centres, so we join by postal code
postal codes used by more than one row of hawker_geo.csv: 0
joined by postal code: 104 of 107
after MANUAL_MATCH: 105 of 107
  not in hawker_geo.csv: 'Blk 4A Woodlands Centre Road' -> gov_only
  not in hawker_geo.csv: 'Blks 1A/ 2A/ 3A Commonwealth Drive' -> gov+hdb
rows where total != cooked + produce: 0
data_source
gov+geo     105
geo_only     24
gov_only      1
gov+hdb       1
hawker centres: 131

Centres with NULL coordinates (left out of proximity until coordinates are supplied):
 hawker_id                  centre_name
        44 Blk 4A Woodlands Centre Road


## Step 5: building-to-hawker distances and proximity
For every building and hawker centre with coordinates:
1. Compute the Haversine straight-line distance.
2. Multiply it by `WALK_RATIO` (1.5) to get `distance_m`, the estimated walking distance, because actual roads and paths are longer than the straight line.

A pair is stored only if its estimated walking distance is under 2,400 m (about 1,600 m in a straight line). Only `distance_m` is stored; the straight-line distance is `distance_m / 1.5` and storing it too would be redundant. The three proximity levels apply to `distance_m` and live in the `proximity_level` lookup table:

| Level | Label | Estimated walking distance |
|---|---|---|
| 1 | Walking distance | 0 – 0.8 km |
| 2 | Nearby | 0.8 – 1.6 km |
| 3 | Accessible | 1.6 – 2.4 km |

Buildings or centres without coordinates stay in their tables and are only skipped here. The ratio is an estimate, not a routed path.

In [7]:
def stage_proximity():
    b = pd.read_csv(OUT / "hdb_building.csv", dtype={"block": str})
    h = pd.read_csv(OUT / "hawker_centre.csv")
    n_b_missing, n_h_missing = int(b["latitude"].isna().sum()), int(h["latitude"].isna().sum())
    # rows without coordinates stay in their tables; they just cannot take part in a distance calculation
    b = b.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    h = h.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

    h_lat = h["latitude"].to_numpy(float)[None, :]
    h_lon = h["longitude"].to_numpy(float)[None, :]
    parts = []
    for start in range(0, len(b), 2000):           # chunks keep the distance matrix small
        chunk = b.iloc[start:start + 2000]
        straight = haversine_m(chunk["latitude"].to_numpy(float)[:, None],
                               chunk["longitude"].to_numpy(float)[:, None], h_lat, h_lon)
        walk = np.round(straight * WALK_RATIO, 1)  # round BEFORE filtering so a value equal to the cutoff cannot slip through
        bi, hi = np.where(walk < MAX_DIST_M)
        parts.append(pd.DataFrame({
            "building_id": chunk["building_id"].to_numpy()[bi],
            "hawker_id": h["hawker_id"].to_numpy()[hi],
            "distance_m": walk[bi, hi],
        }))
    prox = pd.concat(parts, ignore_index=True).sort_values(["building_id", "distance_m"])
    prox.to_csv(OUT / "proximity.csv", index=False)
    print(f"{len(prox):,} building-hawker pairs under {MAX_DIST_M} m estimated walking "
          f"({prox['building_id'].nunique():,} of {len(b):,} located buildings have at least one); "
          f"{n_b_missing} buildings and {n_h_missing} hawker centres skipped for lack of coordinates")


stage_proximity()

30,084 building-hawker pairs under 2400 m estimated walking (9,128 of 9,753 located buildings have at least one); 0 buildings and 1 hawker centres skipped for lack of coordinates


## Step 6: the level lookup table and a check of all seven CSVs
`proximity_level` is a small lookup table, so we write it from the same `LEVELS` setting the analysis uses.

Then each CSV's header is compared with the `\copy` column list in `Task 1/revised/load_data.sql`, the script that loads the CSVs into PostgreSQL. A bulk load therefore cannot fail on a column mismatch.

In [8]:
pd.DataFrame(LEVELS, columns=["level_id", "label", "min_dist_m", "max_dist_m"]).to_csv(
    OUT / "proximity_level.csv", index=False)

# table -> (csv file name, column list) as written in the COPY commands of load_data.sql
copy_cmds = re.findall(r"\\?COPY\s+housing\.(\w+)\s*\(([^)]*)\)\s+FROM\s+'([^']+)'", LOAD_SCRIPT.read_text(), flags=re.IGNORECASE)
TABLE_FILES = {t: (Path(p).name, [c.strip() for c in cols.split(",")]) for t, cols, p in copy_cmds}
print(f"{len(TABLE_FILES)} COPY commands found in {LOAD_SCRIPT.relative_to(ROOT)}\n")

problems = 0
for table, (fname, cols) in TABLE_FILES.items():
    path = OUT / fname
    if not path.exists():
        print(f"MISSING {fname}")
        problems += 1
        continue
    header = list(pd.read_csv(path, nrows=0).columns)
    n_rows = len(pd.read_csv(path, usecols=[0]))
    ok = header == cols
    problems += (not ok)
    print(f"{'OK ' if ok else 'BAD'} {table:20s} <- {fname:24s} {n_rows:>9,} rows")
    if not ok:
        print("    expected:", cols)
        print("    found:   ", header)
print(f"\nAll {len(TABLE_FILES)} table files match the load script." if problems == 0
      else f"\n{problems} problem(s) above.")

7 \copy commands found in Task 1/revised/load_data.sql

OK  town                 <- town.csv                        26 rows
OK  hdb_building         <- hdb_building.csv             9,753 rows
OK  hdb_flat             <- hdb_flat.csv                98,205 rows
OK  resale_transaction   <- resale_transaction.csv     241,483 rows
OK  hawker_centre        <- hawker_centre.csv              131 rows
OK  proximity_level      <- proximity_level.csv              3 rows
OK  proximity            <- proximity.csv               30,084 rows

All 7 table files match the load script.


## Step 7: validate the output
We re-read the CSVs, exactly what the database will receive, and check them against the schema design:
- Primary keys and alternate keys are unique, and foreign keys have no orphans.
- Every Task 1 CHECK constraint holds (text canonical, flat types, centre types, ranges, dates, coordinates).
- Every stored pair falls into exactly one proximity level.
- NULLs are counted for every table.

The report is also saved to `data_quality_report.txt` for the write-up.

In [9]:
def stage_validate():
    lines = []

    def log(s=""):
        print(s)
        lines.append(s)

    rd = lambda n, **kw: pd.read_csv(OUT / n, **kw)  # noqa: E731
    t = rd("town.csv")
    b = rd("hdb_building.csv", dtype={"block": str})
    f = rd("hdb_flat.csv")
    x = rd("resale_transaction.csv")
    h = rd("hawker_centre.csv")
    p = rd("proximity.csv")
    lv = rd("proximity_level.csv")
    tables = [("town", t, ["town_id"]), ("hdb_building", b, ["building_id"]), ("hdb_flat", f, ["flat_id"]),
              ("resale_transaction", x, ["transaction_id"]), ("hawker_centre", h, ["hawker_id"]),
              ("proximity_level", lv, ["level_id"]), ("proximity", p, ["building_id", "hawker_id"])]
    log(f"resale files: {', '.join(Path(r).name for r in RESALE_FILES)}")

    log("\n== rows and primary-key duplicates ==")
    for name, d, pk in tables:
        log(f"{name}: {len(d):,} rows, duplicate keys: {int(d.duplicated(pk).sum())}")

    log("\n== alternate-key duplicates (should all be 0) ==")
    log(f"town.town_name: {int(t.duplicated('town_name').sum())}")
    log(f"hdb_building (block, street_name): {int(b.duplicated(['block', 'street_name']).sum())}")
    log(f"hdb_flat full profile: {int(f.drop(columns='flat_id').duplicated().sum())}")
    log(f"hawker_centre.centre_name: {int(h.duplicated('centre_name').sum())}")
    log(f"proximity_level.label: {int(lv.duplicated('label').sum())}")

    log("\n== foreign-key orphans (should all be 0) ==")
    for child, col, parent, pcol, label in [
            (b, "town_id", t, "town_id", "hdb_building.town_id"),
            (f, "building_id", b, "building_id", "hdb_flat.building_id"),
            (x, "flat_id", f, "flat_id", "resale_transaction.flat_id"),
            (p, "building_id", b, "building_id", "proximity.building_id"),
            (p, "hawker_id", h, "hawker_id", "proximity.hawker_id")]:
        log(f"{label}: {int((~child[col].isin(parent[pcol])).sum())}")

    def not_canonical(s):
        s = s.astype(str)
        return int(((s == "") | (s != s.str.strip().str.upper())).sum())

    tm = pd.to_datetime(x["transaction_month"])
    lease = x["reported_remaining_lease_months"].dropna()
    log("\n== Task 1 CHECK constraints (violations, should all be 0) ==")
    log(f"text not upper-case/trimmed: town_name {not_canonical(t['town_name'])}, block {not_canonical(b['block'])}, "
        f"street_name {not_canonical(b['street_name'])}, flat_model {not_canonical(f['flat_model'])}")
    log(f"flat_type outside the 7 HDB types: {int((~f['flat_type'].isin(FLAT_TYPES)).sum())}")
    log(f"floor_area_sqm <= 0: {int((f['floor_area_sqm'] <= 0).sum())}")
    log(f"storey_lower < 1 or > storey_upper: {int(((f['storey_lower'] < 1) | (f['storey_lower'] > f['storey_upper'])).sum())}")
    log(f"lease_commence_year outside 1960-2100: {int((~f['lease_commence_year'].between(1960, 2100)).sum())}")
    log(f"resale_price <= 0: {int((x['resale_price'] <= 0).sum())}")
    log(f"transaction_month not the 1st or before 1990: {int(((tm.dt.day != 1) | (tm < '1990-01-01')).sum())}")
    log(f"remaining lease outside 0-1188 months: {int((~lease.between(0, 1188)).sum())}")
    log(f"centre_type outside HC/MHC/MK: {int((~h['centre_type'].dropna().isin(['HC', 'MHC', 'MK'])).sum())}")
    s = h[["total_stalls", "cooked_food_stalls", "market_produce_stalls"]]
    log(f"stall count negative or part > total: "
        f"{int(((s < 0).any(axis=1) | (s['cooked_food_stalls'] > s['total_stalls']) | (s['market_produce_stalls'] > s['total_stalls'])).sum())}")
    log(f"building coordinate_source 'none' but has coordinates (or the reverse): "
        f"{int(((b['coordinate_source'] == 'none') != b['latitude'].isna()).sum())}")
    log(f"negative distances: {int((p['distance_m'] < 0).sum())}")

    log("\n== proximity levels ==")
    hits = sum(((p["distance_m"] >= lo) & (p["distance_m"] < hi)).astype(int) for _, _, lo, hi in LEVELS)
    log(f"pairs in no level: {int((hits == 0).sum())}, pairs in more than one level: {int((hits > 1).sum())}")
    for lid, label, lo, hi in LEVELS:
        n = int(((p["distance_m"] >= lo) & (p["distance_m"] < hi)).sum())
        log(f"level {lid} ({label}, {lo / 1000:.1f}-{hi / 1000:.1f} km): {n:,} pairs")
    nearest = p.groupby("building_id")["distance_m"].min()
    log(f"buildings with a hawker centre within {MAX_DIST_M} m walking: {len(nearest):,} of {len(b):,}")
    log("buildings by level of their nearest centre:")
    for lid, label, lo, hi in LEVELS:
        log(f"  level {lid} ({label}): {int(((nearest >= lo) & (nearest < hi)).sum()):,}")
    log(f"  none within {MAX_DIST_M} m: {len(b) - len(nearest):,}")

    log(f"\ntransaction months: {x['transaction_month'].min()} to {x['transaction_month'].max()}")
    log(f"years covered: {tm.dt.year.min()}-{tm.dt.year.max()} ({tm.dt.year.nunique()} years)")

    log("\n== NULL counts (NULL is only expected where no source has the value) ==")
    for name, d, _ in tables:
        n = d.isna().sum()
        n = n[n > 0]
        log(f"{name}: " + (", ".join(f"{k}={int(v)}" for k, v in n.items()) or "none"))

    log("\n== coordinates ==")
    for name, d in (("hdb_building", b), ("hawker_centre", h)):
        c = d.dropna(subset=["latitude", "longitude"])
        outside = ~(c["latitude"].between(*LAT_RANGE) & c["longitude"].between(*LON_RANGE))
        log(f"{name}: {len(d) - len(c)} rows without coordinates (NULL), {int(outside.sum())} outside Singapore")
    log("building coordinate sources:")
    log(b["coordinate_source"].value_counts().to_string())
    log("\n== hawker sources ==")
    log(h["data_source"].value_counts().to_string())
    (OUT / "data_quality_report.txt").write_text("\n".join(lines) + "\n")
    print(f"\nreport written to {(OUT / 'data_quality_report.txt').relative_to(ROOT)}")


stage_validate()

resale files: Resale flat prices based on registration date from Jan-2017 onwards.csv

== rows and primary-key duplicates ==
town: 26 rows, duplicate keys: 0
hdb_building: 9,753 rows, duplicate keys: 0
hdb_flat: 98,205 rows, duplicate keys: 0
resale_transaction: 241,483 rows, duplicate keys: 0
hawker_centre: 131 rows, duplicate keys: 0
proximity_level: 3 rows, duplicate keys: 0
proximity: 30,084 rows, duplicate keys: 0

== alternate-key duplicates (should all be 0) ==
town.town_name: 0
hdb_building (block, street_name): 0
hdb_flat full profile: 0
hawker_centre.centre_name: 0
proximity_level.label: 0

== foreign-key orphans (should all be 0) ==
hdb_building.town_id: 0
hdb_flat.building_id: 0
resale_transaction.flat_id: 0
proximity.building_id: 0
proximity.hawker_id: 0

== Task 1 CHECK constraints (violations, should all be 0) ==
text not upper-case/trimmed: town_name 0, block 0, street_name 0, flat_model 0
flat_type outside the 7 HDB types: 0
floor_area_sqm <= 0: 0
storey_lower < 1 or >

## Step 8: look at the results
Display the first few rows of each table as a final visual check. The CSVs can then be loaded into PostgreSQL with `Task 1/revised/schema.sql` and `Task 1/revised/load_data.sql`.

In [10]:
for name in ["town", "hdb_building", "hdb_flat", "resale_transaction", "hawker_centre", "proximity_level", "proximity"]:
    df = pd.read_csv(OUT / f"{name}.csv")
    print(f"--- {name}: {len(df):,} rows")
    display(df.head(3))

--- town: 26 rows


,town_id,town_name
0,1,ANG MO KIO
1,2,BEDOK
2,3,BISHAN


--- hdb_building: 9,753 rows


,building_id,town_id,block,street_name,latitude,longitude,coordinate_source
0,1,1,101,ANG MO KIO AVE 3,1.370262,103.839468,hdb_dataset
1,2,1,102,ANG MO KIO AVE 3,1.370650,103.839410,hdb_dataset
2,3,1,103,ANG MO KIO AVE 3,1.371017,103.839305,hdb_dataset


--- hdb_flat: 98,205 rows


...

,hawker_id,centre_name,address,centre_type,latitude,longitude,total_stalls,cooked_food_stalls,market_produce_stalls,data_source
0,1,ABC Brickworks Market & Food Centre,"Blk 6, Jalan Bukit Merah, Singapore 150006",MHC,1.286883,103.808131,144.0,96.0,48.0,gov+geo
1,2,Adam Road Food Centre,"2, Adam Road, Singapore 289876",HC,1.324160,103.814166,32.0,32.0,0.0,gov+geo
2,3,Albert Centre Market & Food Centre,"Blk 270, Queen Street, Singapore 180270",MHC,1.301102,103.854116,234.0,86.0,148.0,gov+geo


--- proximity_level: 3 rows


,level_id,label,min_dist_m,max_dist_m
0,1,Walking distance,0,800
1,2,Nearby,800,1600
2,3,Accessible,1600,2400


--- proximity: 30,084 rows


,building_id,hawker_id,distance_m
0,1,28,519.5
1,1,27,579.7
2,1,74,713.2
